# Working with DataFrames API

In [1]:
%pyspark

data = [("Java", "20000"), ("Python", "100000"), ("Scala", "3000")]
rdd = spark.sparkContext.parallelize(data)

df1 = rdd.toDF() # Since RDD doesn’t have columns, the DataFrame is created with default column names “_1” and “_2” as we have two columns
df1.printSchema()


In [2]:
%pyspark
columns = ["language","users_count"]
data = [("Java", 20000), ("Python", 100000), ("Scala", 3000)]

rdd = spark.sparkContext.parallelize(data)    # create RDD from sample data

df = rdd.toDF(columns)  # create DataFrame from RDD and apply column names. Column data types will be generated automatically
df.printSchema()

In [3]:
%pyspark

schema = "language STRING, count INT, flag BOOLEAN"   # DDL-like schema

data = [("Java", 20000, True), ("Python", 100000, False), ("Scala", 3000, True)]

rdd = spark.sparkContext.parallelize(data)     # create RDD from sample data
df = spark.createDataFrame(rdd,schema)      # create DataFrame with schema from RDD. With createDataFrame you have full control over schema customization. 

df.printSchema()
df.show()

schema2 = df.schema     # retrieve schema from DataFrame
print(schema2)

df2 = spark.createDataFrame(rdd,schema2)   # create another datafrmane with new schema
df2.printSchema()

In [4]:
%pyspark

# While working with files, some times we may not receive a file for processing
# If we don’t create with the same schema, our operations/transformations (like union’s) on DF fail as we refer to the columns that may not present.

schema = "language STRING, count INT, flag BOOLEAN"  

empty_rdd = spark.sparkContext.emptyRDD()    # create empty RDD

df = spark.createDataFrame(empty_rdd,schema) # create emoty dataframe 
df.printSchema()
df.show()

In [5]:
%pyspark
schema = "movie STRING, title STRING, genres STRING, new_col STRING"
df = spark.read.options(header='True', inferSchema='False', delimiter=',').schema(schema).csv("gs://oklev-uku-datasets/movie-ratings/movies.csv")
df.printSchema()
df.show()

In [6]:
%pyspark

df.select("*").show()                   # prints in table format
df.select("movie", "genres").take(5)    # outputs to driver as array. same for collect()

In [7]:
%pyspark

from pyspark.sql.functions import col

schema = "movie STRING, title STRING, genres STRING"
df = spark.read.options(header='True', inferSchema='False', delimiter=',').schema(schema).csv("gs://oklev-uku-datasets/movie-ratings/movies.csv")

new_df = df.withColumn("new_movie", col("movie").cast("Integer")*100).withColumnRenamed("movie", "old_movie")
new_df.show()
new_df.printSchema()
#df.withColumn("new",  "movie".cast("Integer") * 10).show()  

In [8]:
%pyspark
from pyspark.sql.functions import col

schema = "movie STRING, title STRING, genres STRING, new_col STRING"
df = spark.read.options(header='True', inferSchema='False', delimiter=',').schema(schema).csv("gs://oklev-uku-datasets/movie-ratings/movies.csv")

df.select("movie", "genres").where(df.movie == "5").show()   
df.select("movie", "genres").where(col("movie") == "5").show() 
df.select("movie", "genres").where("cast(movie as int) > 3 and cast(movie as int) != 5").show()   # this is SQL-like expression and here we have communication with Hive

df.select("movie", "title", "genres").where(df.genres.like("%Comedy%")).show() 
df.select("movie", "title", "genres").where("genres like '%Comedy%'").show() 

In [9]:
%pyspark

schema = "language STRING, count INT, flag BOOLEAN"   # DDL-like schema
data = [("Java", 20000, True), ("Python", 100000, False), ("Scala", 3000, True),  ("Python", 1, True)]

rdd = spark.sparkContext.parallelize(data)
df = spark.createDataFrame(rdd,schema)

df.select("language").distinct().show() 

df2 = df.select("language").dropDuplicates()

df2.show()



In [10]:
%pyspark
simpleData = [("James","Sales","NY",90000,34,10000), \
    ("Michael","Sales","NY",86000,56,20000), \
    ("Robert","Sales","CA",81000,30,23000), \
    ("Maria","Finance","CA",90000,24,23000), \
    ("Raman","Finance","CA",99000,40,24000), \
    ("Scott","Finance","NY",83000,36,19000), \
    ("Jen","Finance","NY",79000,53,15000), \
    ("Jeff","Marketing","CA",80000,25,18000), \
    ("Kumar","Marketing","NY",91000,50,21000) \
  ]
columns= ["employee_name","department","state","salary","age","bonus"]

df = spark.createDataFrame(data = simpleData, schema = columns)
df.printSchema()
df.show(truncate=False)

df.sort("department","state").show(truncate=False)
df.orderBy("department","state").show(truncate=False)   # in place


In [11]:
%pyspark
simpleData = [("James","Sales","NY",90000,34,10000),
    ("Michael","Sales","NY",86000,56,20000),
    ("Robert","Sales","CA",81000,30,23000),
    ("Maria","Finance","CA",90000,24,23000),
    ("Raman","Finance","CA",99000,40,24000),
    ("Scott","Finance","NY",83000,36,19000),
    ("Jen","Finance","NY",79000,53,15000),
    ("Jeff","Marketing","CA",80000,25,18000),
    ("Kumar","Marketing","NY",91000,50,21000)
  ]

schema = ["employee_name","department","state","salary","age","bonus"]
df = spark.createDataFrame(data=simpleData, schema = schema)

df.groupBy("department").sum("salary").show(truncate=False)  #find the sum of salary for each department 

In [12]:
%pyspark
valuesA = [('Pirate',1),('Monkey',2),('Ninja',3),('Spaghetti',4)]
TableA = spark.createDataFrame(valuesA,"name STRING, id INT")
 
valuesB = [('Rutabaga',1),('Pirate',2),('Ninja',3),('Darth Vader',4)]
TableB = spark.createDataFrame(valuesB,"name STRING, id INT")
 
TableA.show()
TableB.show()

ta = TableA.alias('ta')
tb = TableB.alias('tb')

inner_join = ta.join(tb, ta.name == tb.name)
inner_join.show()


# Reading and writing files

In [14]:
%pyspark

schema = "movie STRING, title STRING, genres STRING, new_col STRING"

movies_df = spark.read.options(header='True', inferSchema='False', delimiter=',').schema(schema).csv("gs://oklev-uku-datasets/movie-ratings/movies.csv")

movies_df.printSchema()
movies_df.show()

In [15]:
%pyspark

movies_df = spark.read.options(header='True', inferSchema='True', delimiter=',').csv("gs://oklev-uku-datasets/movie-ratings/movies.csv")

movies_df.printSchema()
movies_df.show()

In [16]:
%pyspark
simpleData = [("James","Sales","NY",90000,34,10000), \
    ("Michael","Sales","NY",86000,56,20000), \
    ("Robert","Sales","CA",81000,30,23000), \
    ("Maria","Finance","CA",90000,24,23000), \
    ("Raman","Finance","CA",99000,40,24000), \
    ("Scott","Finance","NY",83000,36,19000), \
    ("Jen","Finance","NY",79000,53,15000), \
    ("Jeff","Marketing","CA",80000,25,18000), \
    ("Kumar","Marketing","NY",91000,50,21000) \
  ]
columns= ["employee_name","department","state","salary","age","bonus"]

df = spark.createDataFrame(data = simpleData, schema = columns)

df.write.parquet("hdfs:///tmp/people4.parquet")



In [17]:
%sh
hdfs dfs -ls /tmp/people4.parquet

In [18]:
%pyspark
df = spark.read.parquet("hdfs:///tmp/people4.parquet")
df.show()
df.printSchema()

# Using Raw SQL

In [20]:
%pyspark

tables = spark.sql("show tables")      # show Hive tables

tables.show()

movies_df = spark.sql("SELECT movieid, title, genres FROM movies_text")      # movie is a Hive table

movies_df.printSchema()
movies_df.show()

In [21]:
%pyspark

join_df = spark.sql("SELECT UserID, m.MovieID, upper(Title), Genres, if (Rating < 3.0,0.0,Rating) FROM ratings_text r JOIN movies_text m ON r.MovieID=m.MovieID")  

join_df.printSchema()
join_df.show()


In [22]:
%pyspark
simpleData = [("James","Sales","NY",90000,34,10000), \
    ("Michael","Sales","NY",86000,56,20000), \
    ("Robert","Sales","CA",81000,30,23000), \
    ("Maria","Finance","CA",90000,24,23000), \
    ("Raman","Finance","CA",99000,40,24000), \
    ("Scott","Finance","NY",83000,36,19000), \
    ("Jen","Finance","NY",79000,53,15000), \
    ("Jeff","Marketing","CA",80000,25,18000), \
    ("Kumar","Marketing","NY",91000,50,21000) \
  ]
columns= ["employee_name","department","state","salary","age","bonus"]

df = spark.createDataFrame(data = simpleData, schema = columns)

df.createOrReplaceTempView("EMP")  # register temp table

spark.sql("select employee_name,department,state,salary,age,bonus from EMP where salary > 80000 ORDER BY department asc").show(truncate=False)

spark.sql("select department, sum(salary) from EMP GROUP BY department").show(truncate=False)

In [23]:
%pyspark
valuesA = [('Pirate',1),('Monkey',2),('Ninja',3),('Spaghetti',4)]
TableA = spark.createDataFrame(valuesA,"name STRING, id INT")
TableA.createOrReplaceTempView("TableA")
 
valuesB = [('Rutabaga',1),('Pirate',2),('Ninja',3),('Darth Vader',4)]
TableB = spark.createDataFrame(valuesB,"name STRING, id INT")
TableB.createOrReplaceTempView("TableB")

spark.sql("select * from TableA ta join TableB tb on ta.name = tb.name").show()


In [24]:
%pyspark
simpleData = [("James","Sales","NY",90000,34,10000), \
    ("Michael","Sales","NY",86000,56,20000), \
    ("Robert","Sales","CA",81000,30,23000), \
    ("Maria","Finance","CA",90000,24,23000), \
    ("Raman","Finance","CA",99000,40,24000), \
    ("Scott","Finance","NY",83000,36,19000), \
    ("Jen","Finance","NY",79000,53,15000), \
    ("Jeff","Marketing","CA",80000,25,18000), \
    ("Kumar","Marketing","NY",91000,50,21000) \
  ]
columns= ["employee_name","department","state","salary","age","bonus"]

df = spark.createDataFrame(data = simpleData, schema = columns)

df.createOrReplaceTempView("EMP")  # register temp table

spark.sql("create table if not exists EMPLOYEES as select * from EMP")
spark.sql("select * from EMPLOYEES").show()

In [25]:
%sh

# check Spark Web UI
spark-sql \
-d default \
-e "SELECT UserID, m.MovieID, upper(Title), Genres, if (Rating < 3.0,0.0,Rating) FROM ratings_text r JOIN movies_text m ON r.MovieID=m.MovieID"


In [26]:
%pyspark
simpleData = [(1,2)]
columns= ["col1","col2"]

df = spark.createDataFrame(data = simpleData, schema = columns)

df.createOrReplaceTempView("EMP1")  # register temp table

da = spark.sql("insert into df select 1, 3")


In [27]:
%sh
spark-submit --class org.apache.spark.examples.SparkPi \
--conf spark.app.name=spark-pi /usr/lib/spark/examples/jars/spark-examples.jar 10000
